# nb02 Parity: DuckDB SQL and pandas checks against the published Tableau dashboard

**Purpose**
- Reproduce every figure shown on the published `medi_cal_capitation_rates/Dashboard1` in DuckDB SQL and in pandas, independently.
- Compare each figure to the values read off Tableau during the build; any mismatch prints FAIL.

**Prerequisites**
- nb00 and nb01 have been run: `data/clean/` holds the seven `clean_*.csv` files.
- The market share project's enrollment file exists at `../../tableau_la_market_share/data/la_market_share_clean.csv`.
- `duckdb` and `pandas` installed.

**What gets checked (both engines each)**
- Union row count; per file row counts.
- Los Angeles adult working rate by plan and year (16 values).
- Year over year percent change (12 values).
- Statewide FIXED average, before context (6 values, the deliberately wrong ones) and after context (6 values).
- Member months by plan and year (15 values).
- Estimated revenue at the adult rate (15 values, matched to the dashboard's $B rounding).
- The 5% dial scenario for L.A. Care 2025.\n- The category dial families: Child, Maternity, and SPD rates by plan and year (45 values), and the maternity tie (all plans paid one identical rate, every year).

In [ ]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb02_sql_pandas_parity_cell_output.txt"

_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

In [ ]:
# Step 1: load the data both ways
from pathlib import Path
import pandas as pd
import duckdb
import glob

PROJECT_ROOT = Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
ENROLL_CSV = PROJECT_ROOT.parent / "tableau_la_market_share" / "data" / "la_market_share_clean.csv"

files = sorted(glob.glob(str(CLEAN_DIR / "clean_*.csv")))
assert len(files) == 7, f"Expected 7 clean files, found {len(files)}"
rates = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)   # the union, pandas side
enroll = pd.read_csv(ENROLL_CSV)

con = duckdb.connect()
con.execute(f"""
    CREATE VIEW rates AS
    SELECT * FROM read_csv_auto('{CLEAN_DIR.as_posix()}/clean_*.csv', union_by_name=true)
""")
con.execute(f"CREATE VIEW enroll AS SELECT * FROM read_csv_auto('{ENROLL_CSV.as_posix()}')")

# Working Rate exactly as the Tableau calculated field: midpoint, else halfway between the bounds
rates["WR"] = rates["Midpoint"].fillna((rates["Lower Bound"] + rates["Upper Bound"]) / 2)
WR_SQL = 'COALESCE("Midpoint", ("Lower Bound" + "Upper Bound") / 2)'

print(f"pandas union: {len(rates)} rows | enrollment: {len(enroll)} rows")

In [ ]:
# Step 2: the check harness
RESULTS = []

def check(name, got, expected, tol=0.006):
    """Record one comparison; numeric within tol, everything else exact."""
    try:
        ok = abs(float(got) - float(expected)) <= tol
    except (TypeError, ValueError):
        ok = got == expected
    RESULTS.append(ok)
    flag = "PASS" if ok else "FAIL"
    print(f"  [{flag}] {name}: got {got}, expected {expected}")
    return ok

def summary():
    print(f"\n{'='*50}\nTOTAL: {sum(RESULTS)}/{len(RESULTS)} checks passed"
          + ("" if all(RESULTS) else "  <<< INVESTIGATE FAILURES"))

print("Harness ready")

In [ ]:
# Step 3: union and per file row counts
print("Check group 1: row counts")
check("union rows (pandas)", len(rates), 14323)
check("union rows (SQL)", con.execute("SELECT COUNT(*) FROM rates").fetchone()[0], 14323)
for name, n in [("clean_two_plan", 4009), ("clean_cohs", 4478), ("clean_gmc", 1045),
                ("clean_regional", 2393), ("clean_single_plan", 414), ("clean_scan", 108),
                ("clean_pace", 1876)]:
    got = len(pd.read_csv(CLEAN_DIR / f"{name}.csv"))
    check(f"{name} rows", got, n)

In [ ]:
# Step 4: Los Angeles adult working rate by plan and year (the LA Rate Check sheet)
# pandas approach: filter to LA, Two-Plan, the adult categories; pivot Brand x Year on the working rate
# SQL approach: same filter in a WHERE clause, GROUP BY brand and year
EXPECTED_RATE = {
    ("Health Net", 2021): 206.93, ("L.A. Care", 2021): 216.59,
    ("Health Net", 2022): 175.61, ("L.A. Care", 2022): 187.23,
    ("Health Net", 2023): 177.84, ("L.A. Care", 2023): 195.52,
    ("Health Net", 2024): 230.745, ("Kaiser Permanente", 2024): 279.245, ("L.A. Care", 2024): 244.48,
    ("Health Net", 2025): 246.865, ("Kaiser Permanente", 2025): 301.30, ("L.A. Care", 2025): 265.25,
    ("Health Net", 2026): 270.505, ("Kaiser Permanente", 2026): 321.135, ("L.A. Care", 2026): 293.70,
}
ADULT = ["Adult", "Adult - SIS"]
la = rates[(rates["County"] == "Los Angeles") & (rates["Model"] == "Two-Plan")
           & (rates["Category of Aid"].isin(ADULT))]
pd_rate = la.groupby(["Brand", "Calendar Year"])["WR"].sum()

sql_rate = con.execute(f"""
    SELECT "Brand", "Calendar Year" AS yr, SUM({WR_SQL}) AS wr
    FROM rates
    WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan'
      AND "Category of Aid" IN ('Adult', 'Adult - SIS')
    GROUP BY 1, 2
""").df().set_index(["Brand", "yr"])["wr"]

print("Check group 2: LA adult working rate, pandas")
for (b, y), exp in EXPECTED_RATE.items():
    check(f"pandas {b} {y}", round(pd_rate.loc[(b, y)], 3), exp)
print("Check group 2: LA adult working rate, SQL")
for (b, y), exp in EXPECTED_RATE.items():
    check(f"SQL {b} {y}", round(sql_rate.loc[(b, y)], 3), exp)

In [ ]:
# Step 5: year over year percent change (the middle dashboard table)
# pandas approach: pct_change within each plan down the years
# SQL approach: a window function, LAG over year within each plan
EXPECTED_YOY = {
    ("Health Net", 2022): -15.14, ("Health Net", 2023): 1.27, ("Health Net", 2024): 29.75,
    ("Health Net", 2025): 6.99, ("Health Net", 2026): 9.58,
    ("L.A. Care", 2022): -13.56, ("L.A. Care", 2023): 4.43, ("L.A. Care", 2024): 25.04,
    ("L.A. Care", 2025): 8.50, ("L.A. Care", 2026): 10.73,
    ("Kaiser Permanente", 2025): 7.90, ("Kaiser Permanente", 2026): 6.58,
}
pd_yoy = (pd_rate.unstack("Calendar Year").T.pct_change() * 100).T.stack()

sql_yoy = con.execute(f"""
    WITH r AS (
        SELECT "Brand", "Calendar Year" AS yr, SUM({WR_SQL}) AS wr
        FROM rates
        WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan'
          AND "Category of Aid" IN ('Adult', 'Adult - SIS')
        GROUP BY 1, 2
    )
    SELECT "Brand", yr,
           (wr - LAG(wr) OVER (PARTITION BY "Brand" ORDER BY yr))
           / LAG(wr) OVER (PARTITION BY "Brand" ORDER BY yr) * 100 AS pct
    FROM r
""").df().dropna().set_index(["Brand", "yr"])["pct"]

print("Check group 3: year over year percent, pandas")
for (b, y), exp in EXPECTED_YOY.items():
    check(f"pandas {b} {y}", round(pd_yoy.loc[(b, y)], 2), exp)
print("Check group 3: year over year percent, SQL")
for (b, y), exp in EXPECTED_YOY.items():
    check(f"SQL {b} {y}", round(sql_yoy.loc[(b, y)], 2), exp)

In [ ]:
# Step 6: the statewide FIXED average, before and after context
# pandas approach: before = mean over the whole union per year; after = mean within Two-Plan adult rows only
# SQL approach: same two aggregations, no county restriction in either (that is the FIXED behavior)
EXPECTED_PRE  = {2021: 2763.81, 2022: 3138.56, 2023: 2585.78, 2024: 2848.31, 2025: 1860.22, 2026: 1986.15}
EXPECTED_POST = {2021: 256.28, 2022: 221.87, 2023: 217.84, 2024: 290.48, 2025: 305.54, 2026: 337.14}

pd_pre = rates.groupby("Calendar Year")["WR"].mean()
pd_post = rates[(rates["Model"] == "Two-Plan") & (rates["Category of Aid"].isin(ADULT))] \
    .groupby("Calendar Year")["WR"].mean()

sql_pre = con.execute(f'SELECT "Calendar Year" AS yr, AVG({WR_SQL}) FROM rates GROUP BY 1').df().set_index("yr").iloc[:, 0]
sql_post = con.execute(f"""
    SELECT "Calendar Year" AS yr, AVG({WR_SQL})
    FROM rates
    WHERE "Model" = 'Two-Plan' AND "Category of Aid" IN ('Adult', 'Adult - SIS')
    GROUP BY 1
""").df().set_index("yr").iloc[:, 0]

print("Check group 4: statewide average BEFORE context (the deliberately wrong one)")
for y, exp in EXPECTED_PRE.items():
    check(f"pandas pre {y}", round(pd_pre.loc[y], 2), exp)
    check(f"SQL pre {y}", round(sql_pre.loc[y], 2), exp)
print("Check group 4: statewide average AFTER context")
for y, exp in EXPECTED_POST.items():
    check(f"pandas post {y}", round(pd_post.loc[y], 2), exp)
    check(f"SQL post {y}", round(sql_post.loc[y], 2), exp)

In [ ]:
# Step 7: member months by plan and year
# pandas approach: sum the monthly member counts within each plan and year
# SQL approach: SUM with GROUP BY, 2021 onward
EXPECTED_MM = {
    ("Health Net", 2021): 11862307, ("Health Net", 2022): 12799596, ("Health Net", 2023): 14053188,
    ("Health Net", 2024): 14213432, ("Health Net", 2025): 14245664, ("Health Net", 2026): 6697296,
    ("Kaiser Permanente", 2024): 3542469, ("Kaiser Permanente", 2025): 3899118, ("Kaiser Permanente", 2026): 2032645,
    ("L.A. Care", 2021): 27185468, ("L.A. Care", 2022): 29675389, ("L.A. Care", 2023): 32297319,
    ("L.A. Care", 2024): 28364683, ("L.A. Care", 2025): 28266681, ("L.A. Care", 2026): 13117088,
}
pd_mm = enroll[enroll["Year"] >= 2021].groupby(["Brand", "Year"])["Enrollees"].sum()
sql_mm = con.execute("""
    SELECT "Brand", "Year" AS yr, SUM("Enrollees") AS mm
    FROM enroll WHERE "Year" >= 2021 GROUP BY 1, 2
""").df().set_index(["Brand", "yr"])["mm"]

print("Check group 5: member months")
for (b, y), exp in EXPECTED_MM.items():
    check(f"pandas {b} {y}", int(pd_mm.loc[(b, y)]), exp, tol=0.5)
    check(f"SQL {b} {y}", int(sql_mm.loc[(b, y)]), exp, tol=0.5)

In [ ]:
# Step 8: estimated revenue at the adult rate, matched to the dashboard's $B rounding
# pandas approach: member months x working rate per plan and year, shown in billions to 1 decimal
# SQL approach: join the two aggregates on plan and year, same arithmetic
EXPECTED_REV_B = {
    ("Health Net", 2021): 2.5, ("Health Net", 2022): 2.2, ("Health Net", 2023): 2.5,
    ("Health Net", 2024): 3.3, ("Health Net", 2025): 3.5, ("Health Net", 2026): 1.8,
    ("Kaiser Permanente", 2024): 1.0, ("Kaiser Permanente", 2025): 1.2, ("Kaiser Permanente", 2026): 0.7,
    ("L.A. Care", 2021): 5.9, ("L.A. Care", 2022): 5.6, ("L.A. Care", 2023): 6.3,
    ("L.A. Care", 2024): 6.9, ("L.A. Care", 2025): 7.5, ("L.A. Care", 2026): 3.9,
}
pd_rev = (pd_mm * pd_rate.rename_axis(["Brand", "Year"])).dropna() / 1e9

sql_rev = con.execute(f"""
    WITH r AS (
        SELECT "Brand", "Calendar Year" AS yr, SUM({WR_SQL}) AS wr
        FROM rates
        WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan'
          AND "Category of Aid" IN ('Adult', 'Adult - SIS')
        GROUP BY 1, 2
    ), m AS (
        SELECT "Brand", "Year" AS yr, SUM("Enrollees") AS mm
        FROM enroll WHERE "Year" >= 2021 GROUP BY 1, 2
    )
    SELECT r."Brand", r.yr, m.mm * r.wr / 1e9 AS rev_b
    FROM r JOIN m ON r."Brand" = m."Brand" AND r.yr = m.yr
""").df().set_index(["Brand", "yr"])["rev_b"]

print("Check group 6: estimated revenue ($B, dashboard rounding)")
for (b, y), exp in EXPECTED_REV_B.items():
    check(f"pandas {b} {y}", round(pd_rev.loc[(b, y)], 1), exp)
    check(f"SQL {b} {y}", round(sql_rev.loc[(b, y)], 1), exp)

In [ ]:
# Step 9: the 5% dial scenario, L.A. Care 2025
# pandas approach: scenario minus baseline at +5% = 5% of baseline revenue
# SQL approach: the same product, in one expression
base = pd_rev.loc[("L.A. Care", 2025)] * 1e9
pd_change_m = base * 0.05 / 1e6
sql_change_m = con.execute(f"""
    WITH r AS (
        SELECT SUM({WR_SQL}) AS wr FROM rates
        WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan' AND "Brand" = 'L.A. Care'
          AND "Calendar Year" = 2025 AND "Category of Aid" IN ('Adult', 'Adult - SIS')
    ), m AS (
        SELECT SUM("Enrollees") AS mm FROM enroll WHERE "Brand" = 'L.A. Care' AND "Year" = 2025
    )
    SELECT m.mm * r.wr * 0.05 / 1e6 FROM r, m
""").fetchone()[0]

print("Check group 7: what a 5% rate move is worth to L.A. Care in 2025 (millions)")
check("pandas 5% scenario ($M)", round(pd_change_m, 0), 375, tol=1)
check("SQL 5% scenario ($M)", round(sql_change_m, 0), 375, tol=1)

In [ ]:
# Step 9b: the category dial families (Child, Maternity, SPD), and the maternity tie
# pandas approach: map the 2023 label split onto stable families (strip ' - SIS', exclude UIS), then pivot
# SQL approach: REPLACE for the mapping, NOT LIKE for the exclusion, same aggregation
EXPECTED_FAMILY = {
    ("Child", "Health Net"):        {2021: 75.61, 2022: 73.27, 2023: 79.11, 2024: 110.445, 2025: 129.885, 2026: 135.27},
    ("Child", "Kaiser Permanente"): {2024: 135.905, 2025: 167.875, 2026: 189.28},
    ("Child", "L.A. Care"):         {2021: 81.10, 2022: 78.99, 2023: 92.78, 2024: 114.235, 2025: 134.605, 2026: 141.18},
    ("Maternity", "Health Net"):        {2021: 7575.48, 2022: 7956.28, 2023: 7077.68, 2024: 8655.405, 2025: 8782.29, 2026: 7962.505},
    ("Maternity", "Kaiser Permanente"): {2024: 8655.405, 2025: 8782.29, 2026: 7962.505},
    ("Maternity", "L.A. Care"):         {2021: 7575.48, 2022: 7956.28, 2023: 7077.68, 2024: 8655.405, 2025: 8782.29, 2026: 7962.505},
    ("SPD", "Health Net"):        {2021: 668.20, 2022: 573.23, 2023: 538.71, 2024: 980.03, 2025: 1115.425, 2026: 1167.045},
    ("SPD", "Kaiser Permanente"): {2024: 816.08, 2025: 897.72, 2026: 1086.40},
    ("SPD", "L.A. Care"):         {2021: 807.01, 2022: 691.05, 2023: 675.77, 2024: 1074.75, 2025: 1232.33, 2026: 1306.795},
}

fam = rates[(rates["County"] == "Los Angeles") & (rates["Model"] == "Two-Plan")
            & (~rates["Category of Aid"].str.contains("UIS", na=False))].copy()
fam["Family"] = fam["Category of Aid"].str.replace(" - SIS", "", regex=False)
pd_fam = fam.groupby(["Family", "Brand", "Calendar Year"])["WR"].sum()

sql_fam = con.execute(f"""
    SELECT REPLACE("Category of Aid", ' - SIS', '') AS family, "Brand", "Calendar Year" AS yr,
           SUM({WR_SQL}) AS wr
    FROM rates
    WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan'
      AND "Category of Aid" NOT LIKE '%UIS%'
    GROUP BY 1, 2, 3
""").df().set_index(["family", "Brand", "yr"])["wr"]

print("Check group 8: category family rates, pandas then SQL")
for (f, b), years in EXPECTED_FAMILY.items():
    for y, exp in years.items():
        check(f"pandas {f} {b} {y}", round(pd_fam.loc[(f, b, y)], 3), exp)
for (f, b), years in EXPECTED_FAMILY.items():
    for y, exp in years.items():
        check(f"SQL {f} {b} {y}", round(sql_fam.loc[(f, b, y)], 3), exp)

print("Check group 8b: the maternity tie (one distinct rate across all plans, every year)")
pd_tie = fam[fam["Family"] == "Maternity"].groupby("Calendar Year")["WR"].nunique()
sql_tie = con.execute(f"""
    SELECT "Calendar Year" AS yr, COUNT(DISTINCT {WR_SQL}) AS n
    FROM rates
    WHERE "County" = 'Los Angeles' AND "Model" = 'Two-Plan'
      AND REPLACE("Category of Aid", ' - SIS', '') = 'Maternity'
      AND "Category of Aid" NOT LIKE '%UIS%'
    GROUP BY 1
""").df().set_index("yr")["n"]
for y in [2021, 2022, 2023, 2024, 2025, 2026]:
    check(f"pandas maternity distinct rates {y}", int(pd_tie.loc[y]), 1, tol=0)
    check(f"SQL maternity distinct rates {y}", int(sql_tie.loc[y]), 1, tol=0)

In [ ]:
# Step 10: verdict
summary()

In [ ]:
# Final step: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Attach `nb02_sql_pandas_parity_cell_output.txt` in the chat.
- A full pass clears every figure for the blog page; any FAIL stops the post until resolved.